# Assault DDQN - HU008 MLflow experiment tracking


## 1. Bootstrap Local -> GitHub -> Colab

In [ ]:
import os


os.environ.setdefault("ASSAULT_BOOTSTRAP_REF", "main")
os.environ.setdefault("ASSAULT_E2E_REQUIRE_CUDA", "1")


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/j-mauro-r/reinforcement_learning_reto_1.git"
COLAB_ROOT = Path("/content/reinforcement_learning_reto_1")
BOOTSTRAP_REF = os.environ.get("ASSAULT_BOOTSTRAP_REF", "main")
BOOTSTRAP_COMMIT = os.environ.get("ASSAULT_BOOTSTRAP_COMMIT") or None
INSTALL_DEPENDENCIES = os.environ.get("ASSAULT_INSTALL_DEPENDENCIES", "1") == "1"


def _running_in_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return False
    return True


def _git_output(args, cwd):
    return subprocess.check_output(["git", *args], cwd=str(cwd), text=True).strip()


if _running_in_colab():
    if not (COLAB_ROOT / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_ROOT)], check=True)
    subprocess.run(["git", "fetch", "--prune", "origin"], cwd=str(COLAB_ROOT), check=True)
    provisional_ref = BOOTSTRAP_COMMIT or f"origin/{BOOTSTRAP_REF}"
    provisional_sha = _git_output(["rev-parse", "--verify", f"{provisional_ref}^{{commit}}"], COLAB_ROOT)
    subprocess.run(["git", "checkout", "--detach", provisional_sha], cwd=str(COLAB_ROOT), check=True)
    ASSAULT_DIR = COLAB_ROOT / "2_Assault"
else:
    PROJECT_ROOT = Path(_git_output(["rev-parse", "--show-toplevel"], Path.cwd()))
    ASSAULT_DIR = PROJECT_ROOT / "2_Assault"

for path in (ASSAULT_DIR, ASSAULT_DIR.parent):
    value = str(path.resolve())
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

from src.execution_bootstrap import (
    install_project_requirements,
    prepare_execution_environment,
    verify_environment_import,
)

bootstrap = prepare_execution_environment(
    requested_ref=BOOTSTRAP_REF,
    requested_commit=BOOTSTRAP_COMMIT,
    repo_url=REPO_URL,
    colab_root=COLAB_ROOT,
)

PROJECT_ROOT = bootstrap.repo_root
ASSAULT_DIR = bootstrap.assault_dir

if INSTALL_DEPENDENCIES:
    install_project_requirements(bootstrap.requirements_path)

environment_source = verify_environment_import(bootstrap)
bootstrap.as_dict()


In [ ]:
from dataclasses import fields
from src.e2e_smoke import E2ESmokeSummary
import src.e2e_smoke as e2e_smoke_module

print(e2e_smoke_module.__file__)
print([field.name for field in fields(E2ESmokeSummary)])

## 2. Imports, configuration and tracking


In [ ]:
from pathlib import Path

from src.agent import DDQNAgent
from src.callbacks import TensorBoardLogger, load_tensorboard_scalars
from src.checkpointing import CheckpointManager, reconstruct_epsilon
from src.e2e_smoke import run_e2e_smoke
from src.environment import create_assault_env, get_environment_metadata, validate_frameskip_once
from src.preflight import run_preflight_checks
from src.replay_buffer import ReplayBuffer
from src.tracking import MLflowTracker
from src.trainer import Trainer
from src.utils import get_runtime_info, load_yaml_config

config = load_yaml_config(ASSAULT_DIR / "configs" / "ddqn_config.yaml")
seed = int(config["reproducibility"]["seed"])
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ASSAULT_DIR:", ASSAULT_DIR)
print("BOOTSTRAP_REF:", BOOTSTRAP_REF)
print("BOOTSTRAP_COMMIT:", BOOTSTRAP_COMMIT or "<none>")
print("EXECUTED_SHA:", bootstrap.resolved_sha)
print("src.environment:", environment_source)
config


## 3. Runtime and hardware

In [ ]:
runtime_info = get_runtime_info()
runtime_info


## 4. HU002 environment contract

In [ ]:
train_env = create_assault_env(config, mode="train", seed=seed)
eval_env = create_assault_env(config, mode="eval", seed=seed + 1)

obs, info = train_env.reset(seed=seed)
metadata = get_environment_metadata(train_env, config, mode="train", seed=seed)

print("Observation shape:", obs.shape)
print("Observation dtype:", obs.dtype)
print("Action space:", train_env.action_space)
print("Action meanings:", train_env.unwrapped.get_action_meanings())
print("Initial info:", info)
print("Metadata:", metadata)


## 5. HU002 autovalidations

In [ ]:
assert obs.shape == (4, 84, 84)
assert str(obs.dtype) == "uint8"
assert train_env.action_space.n == 7
assert train_env.observation_space.shape == eval_env.observation_space.shape
assert train_env.observation_space.dtype == eval_env.observation_space.dtype
assert validate_frameskip_once(train_env, expected_frameskip=4, steps=5)

obs, info = train_env.reset(seed=seed)
for step in range(100):
    action = int(train_env.action_space.sample())
    obs, reward, terminated, truncated, info = train_env.step(action)
    assert obs.shape == (4, 84, 84)
    assert str(obs.dtype) == "uint8"
    if terminated or truncated:
        obs, info = train_env.reset()

print("HU002 validations passed.")
train_env.close()
eval_env.close()


## 6. HU004 preflight gate

In [ ]:
preflight_report = run_preflight_checks(config)
print(preflight_report.format_summary())
preflight_report.as_dict()


## 7. Abort if preflight fails

In [ ]:
if not preflight_report.ready_for_training:
    raise RuntimeError("READY_FOR_TRAINING=False; HU005 training aborted.")
print("READY_FOR_TRAINING=True")


## 8. HU007 E2E smoke configuration


In [ ]:
def _env_flag(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y"}

e2e_config = config.get("e2e_smoke", {})
mlflow_config = config.get("mlflow", {})
RUN_ID = os.environ.get("ASSAULT_RUN_ID", config["checkpointing"]["run_id"] + "_hu008_smoke")
CHECKPOINT_DIR = Path(os.environ.get("ASSAULT_CHECKPOINT_DIR", str(ASSAULT_DIR / config["checkpointing"]["directory"])))
TENSORBOARD_DIR = Path(os.environ.get("ASSAULT_TENSORBOARD_DIR", str(ASSAULT_DIR / config.get("tensorboard", {}).get("directory", "logs/tensorboard"))))
E2E_REQUIRE_CUDA = _env_flag("ASSAULT_E2E_REQUIRE_CUDA", bool(e2e_config.get("require_cuda", True)))
MLFLOW_TRACKING_MODE = os.environ.get("ASSAULT_MLFLOW_TRACKING_MODE", mlflow_config.get("tracking_mode", "new")).strip().lower()
MLFLOW_RUN_ID = os.environ.get("ASSAULT_MLFLOW_RUN_ID") or mlflow_config.get("mlflow_run_id")

print("HU008 project_run_id:", RUN_ID)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("TensorBoard directory:", TENSORBOARD_DIR)
print("Segment A timesteps:", e2e_config.get("segment_a_timesteps"))
print("Final timesteps:", e2e_config.get("final_timesteps"))
print("Evaluation episodes:", e2e_config.get("evaluation_episodes"))
print("Evaluation epsilon:", e2e_config.get("evaluation_epsilon"))
print("Require CUDA:", E2E_REQUIRE_CUDA)
print("MLflow enabled:", mlflow_config.get("enabled"))
print("MLflow tracking mode:", MLFLOW_TRACKING_MODE)
print("MLflow configured URI:", os.environ.get("ASSAULT_MLFLOW_TRACKING_URI") or mlflow_config.get("tracking_uri") or mlflow_config.get("local_directory"))
print("MLflow experiment:", os.environ.get("ASSAULT_MLFLOW_EXPERIMENT") or mlflow_config.get("experiment_name"))
print("MLflow resume run id:", MLFLOW_RUN_ID or "<new run>")


## 9. HU008 tracked end-to-end smoke


In [ ]:
MLFLOW_TRACKING_PASS = False
mlflow_tracker = MLflowTracker.from_config(config)
mlflow_metadata = mlflow_tracker.start_run(
    project_run_id=RUN_ID,
    tracking_mode=MLFLOW_TRACKING_MODE,
    mlflow_run_id=MLFLOW_RUN_ID,
    run_name=RUN_ID,
    tags={"stage": "HU008"},
)
print("MLflow tracking URI:", mlflow_metadata.tracking_uri)
print("MLflow experiment name:", mlflow_metadata.experiment_name)
print("project_run_id:", mlflow_metadata.project_run_id)
print("mlflow_run_id:", mlflow_metadata.mlflow_run_id)

try:
    mlflow_tracker.log_run_context(
        config=config,
        runtime_info=runtime_info,
        git_commit=bootstrap.resolved_sha,
        git_ref=BOOTSTRAP_REF,
        project_run_id=RUN_ID,
        action_space=str(metadata.action_space),
        observation_dtype=str(obs.dtype),
        runtime="Google Colab" if _running_in_colab() else "local",
        device="cuda" if __import__("torch").cuda.is_available() else "cpu",
    )
    mlflow_tracker.log_config_snapshot(config)
    mlflow_tracker.log_runtime_metadata(
        runtime_info=runtime_info,
        git_commit=bootstrap.resolved_sha,
        runtime="Google Colab" if _running_in_colab() else "local",
    )

    e2e_summary = run_e2e_smoke(
        config=config,
        checkpoint_root=CHECKPOINT_DIR,
        tensorboard_root=TENSORBOARD_DIR,
        run_id=RUN_ID,
        repo_path=PROJECT_ROOT,
        require_cuda=E2E_REQUIRE_CUDA,
    )
    e2e_result = e2e_summary.as_dict()
    mlflow_tracker.log_training_summary(e2e_summary.segment_b)
    mlflow_tracker.log_evaluation_summary(e2e_summary.evaluation)
    mlflow_tracker.log_checkpoint_reference(
        e2e_summary.checkpoint,
        resume_mode="resume_full",
        project_run_id=RUN_ID,
    )
    mlflow_tracker.log_dict_artifact(e2e_result, "summaries/e2e_smoke_summary.json")
    queried_run = mlflow_tracker.get_run(mlflow_metadata.mlflow_run_id) if mlflow_metadata.enabled else None
    MLFLOW_TRACKING_PASS = bool(
        not mlflow_metadata.enabled
        or (
            queried_run is not None
            and queried_run.info.run_id == mlflow_metadata.mlflow_run_id
            and queried_run.data.params.get("identity.project_run_id") == RUN_ID
            and "train/final_global_step" in queried_run.data.metrics
            and "eval/mean_reward" in queried_run.data.metrics
        )
    )
finally:
    mlflow_tracker.end_run(status="FINISHED" if MLFLOW_TRACKING_PASS else "FAILED")

e2e_result


## 10. HU008 result gates


In [ ]:
assert e2e_summary.preflight.ready_for_training
assert e2e_summary.segment_a.global_step == int(e2e_config["segment_a_timesteps"])
assert e2e_summary.restored.global_step == e2e_summary.checkpoint.checkpoint_step
assert e2e_summary.segment_b.global_step == int(e2e_config["final_timesteps"])
assert e2e_summary.tensorboard_previous_logs_preserved
assert e2e_summary.online_unchanged_during_evaluation
assert e2e_summary.target_unchanged_during_evaluation
assert e2e_summary.optimizer_unchanged_during_evaluation
assert e2e_summary.replay_buffer_unchanged_during_evaluation
assert e2e_summary.training_global_step_unchanged_during_evaluation
if mlflow_config.get("enabled", False):
    assert MLFLOW_TRACKING_PASS, "MLflow tracking validation did not pass."

if e2e_summary.runtime == "Google Colab" and E2E_REQUIRE_CUDA:
    assert e2e_summary.e2e_smoke_pass, "Colab GPU smoke did not pass."
else:
    assert not e2e_summary.e2e_smoke_pass, "E2E_SMOKE_PASS must remain false outside Colab GPU."

print("HU008 MLflow tracking status")
print("runtime:", e2e_summary.runtime)
print("device:", e2e_summary.device)
print("project_run_id:", mlflow_metadata.project_run_id)
print("mlflow_run_id:", mlflow_metadata.mlflow_run_id)
print("mlflow_tracking_uri:", mlflow_metadata.tracking_uri)
print("mlflow_experiment:", mlflow_metadata.experiment_name)
print("observation:", e2e_summary.observation_shape, e2e_summary.observation_dtype)
print("action_space:", e2e_summary.action_space)
print("Preflight READY_FOR_TRAINING:", e2e_summary.preflight.ready_for_training)
print("segment_a_steps:", e2e_summary.segment_a.global_step)
print("segment_a_updates:", e2e_summary.segment_a.updates_count)
print("segment_b_initial_step:", e2e_summary.segment_b.initial_global_step)
print("segment_b_final_step:", e2e_summary.segment_b.global_step)
print("checkpoint_path:", e2e_summary.checkpoint.path)
print("checkpoint_size_bytes:", e2e_summary.checkpoint.size_bytes)
print("tensorboard_event_files:", e2e_summary.tensorboard_event_files_after)
print("evaluation_rewards:", e2e_summary.evaluation.rewards)
print("evaluation_mean_reward:", e2e_summary.evaluation.mean_reward)
print("LOCAL_E2E_SMOKE_PASS=", e2e_summary.local_e2e_smoke_pass)
print("E2E_SMOKE_PASS=", e2e_summary.e2e_smoke_pass)
print("MLFLOW_TRACKING_PASS=", MLFLOW_TRACKING_PASS)
